In [ ]:
import sympy as sp

# 1. Define real, positive symbols
O, dO, e, a = sp.symbols('O dO e a', real=True, positive=True)
# dO represents the discrete phase lag Delta O

# Define the invariant shape factor (e_Y^2 = 1 - e^2)
e_Y2 = 1 - e**2

# 2. Define the exact dynamic radii (Reception and Retarded Emission)
r_A = a * e_Y2 / (1 + e * sp.cos(O))
# Body B is opposite to A (phase O + pi), but delayed by dO.
# Phase is (O + pi - dO). Since cos(x + pi) = -cos(x), the denominator sign flips:
r_B_ret = a * e_Y2 / (1 - e * sp.cos(O - dO))

# 3. The Law of Cosines for the internal signal baseline D^2
# The angle between them is (pi - dO), so cos(pi - dO) = -cos(dO).
# User's exact identity: r_A^2 + r_B^2 - 2*r_A*r_B*(-cos(dO))
D_sq = r_A**2 + r_B_ret**2 + 2 * r_A * r_B_ret * sp.cos(dO)

# 4. Algebraic Order Expansion
# We expand D^2 into a polynomial series with respect to the phase lag (dO)
# to isolate symmetric (bound) vs asymmetric (unclosed) terms.
D_sq_expanded = sp.series(D_sq, dO, 0, 4).removeO()

# Extract the individual geometric components by their order of dO
D_order_0 = D_sq_expanded.subs(dO, 0)
D_order_1 = sp.diff(D_sq_expanded, dO, 1).subs(dO, 0) * dO
D_order_2 = sp.diff(D_sq_expanded, dO, 2).subs(dO, 0) / 2 * dO**2
D_order_3 = sp.diff(D_sq_expanded, dO, 3).subs(dO, 0) / 6 * dO**3

print("--- RELATIONAL BASELINE EXPANSION ---")
print(f"0th Order (Static Baseline): \n{sp.simplify(D_order_0)}\n")
print(f"1st Order (Symmetric Linear Lag, cancels on full orbit): \n{sp.simplify(D_order_1)}\n")
print(f"2nd Order (Symmetric Quadratic Lag, causes orbital precession): \n{sp.simplify(D_order_2)}\n")
print(f"3rd Order (Asymmetric Cubic Lag, the unclosed remainder): \n{sp.simplify(D_order_3)}\n")

--- RELATIONAL BASELINE EXPANSION ---
0th Order (Static Baseline): 
4*a**2*(e**4 - 2*e**2 + 1)/((e*cos(O) - 1)**2*(e*cos(O) + 1)**2)

1st Order (Symmetric Linear Lag, cancels on full orbit): 
-4*a**2*dO*e*(e**2 - 1)**2*sin(O)/((e*cos(O) - 1)**3*(e*cos(O) + 1))

2nd Order (Symmetric Quadratic Lag, causes orbital precession): 
a**2*dO**2*(e**2 - 1)**2*(e**3*cos(O) + 6*e**2*sin(O)**2 - e**2 + e*cos(O) - 1)/((e*cos(O) - 1)**4*(e*cos(O) + 1))

3rd Order (Asymmetric Cubic Lag, the unclosed remainder): 
-a**2*dO**3*e*(e**2 - 1)**2*(6*e**3*cos(O) + 17*e**2*sin(O)**2 + e**2 - 2*e*cos(O) - 5)*sin(O)/(3*(e*cos(O) - 1)**5*(e*cos(O) + 1))



In [ ]:
import sympy as sp
import numpy as np
from scipy.integrate import quad

print("--- 1. EXACT NUMERICAL INTEGRATION (SCIPY) ---")
def integrand(theta, e_val):
    # Pure geometric variables factored out from constants
    e_Y = np.sqrt(1 - e_val**2)

    # 3rd-order spatial asymmetry (geometric part)
    num_D3 = e_val * (e_val**2 - 1)**2 * (6*e_val**3*np.cos(theta) + 17*e_val**2*np.sin(theta)**2 + e_val**2 - 2*e_val*np.cos(theta) - 5) * np.sin(theta)
    den_D3 = 3 * (e_val*np.cos(theta) - 1)**5 * (e_val*np.cos(theta) + 1)
    D3_geom = -num_D3 / den_D3

    # Phase lag (beta_T geometry)
    beta_T_geom = (1 + e_val * np.cos(theta)) / e_Y

    # Potential fluctuation ( (a/r_o)^3 geometry )
    r_o_inv3 = ((1 + e_val * np.cos(theta)) / (1 - e_val**2))**3

    # Radial kinematic projection (beta_R geometry)
    beta_R_geom = e_val * np.sin(theta) / e_Y

    # The total geometric mismatch integrand
    return -0.5 * r_o_inv3 * (D3_geom * beta_T_geom**3) * beta_R_geom

# Execute for Hulse-Taylor
e_HT = 0.6171334
integral_HT, error = quad(integrand, 0, 2*np.pi, args=(e_HT,))
print(f"Hulse-Taylor Geometric Deficit Factor (e = {e_HT}): {integral_HT:.6f}")


print("\n--- 2. NATIVE POLYNOMIAL EXPANSION (SYMPY) ---")
# Define symbols
O, e_sym = sp.symbols('O e', real=True)
e_Y_sym = sp.sqrt(1 - e_sym**2)

# Build the exact symbolic integrand
D3_sym = -e_sym * (e_sym**2 - 1)**2 * (6*e_sym**3*sp.cos(O) + 17*e_sym**2*sp.sin(O)**2 + e_sym**2 - 2*e_sym*sp.cos(O) - 5) * sp.sin(O) / (3*(e_sym*sp.cos(O) - 1)**5 * (e_sym*sp.cos(O) + 1))
beta_T_sym = (1 + e_sym * sp.cos(O)) / e_Y_sym
r_o_inv3_sym = ((1 + e_sym * sp.cos(O)) / (1 - e_sym**2))**3
beta_R_sym = e_sym * sp.sin(O) / e_Y_sym

integrand_sym = -0.5 * r_o_inv3_sym * (D3_sym * beta_T_sym**3) * beta_R_sym

# Taylor expand the integrand to O(e^4) to bypass the rational trig trap
integrand_series = sp.series(integrand_sym, e_sym, 0, 4).removeO()

# Integrate the series term by term
cumulative_series = sp.integrate(integrand_series, (O, 0, 2*sp.pi))
print("Native R.O.M. Polynomial Expansion (Low Eccentricity):")
print(sp.simplify(cumulative_series))

--- 1. EXACT NUMERICAL INTEGRATION (SCIPY) ---
Hulse-Taylor Geometric Deficit Factor (e = 0.6171334): 82.469532

--- 2. NATIVE POLYNOMIAL EXPANSION (SYMPY) ---
Native R.O.M. Polynomial Expansion (Low Eccentricity):
0.833333333333333*pi*e**2


In [ ]:
import sympy as sp

# 1. Define symbolic variables
O, e = sp.symbols('O e', real=True, positive=True)
e_Y = sp.sqrt(1 - e**2)

# 2. Define the exact geometric shape factors
# kappa^2 is proportional to 1/r_o
kappa_sq_shape = (1 + e * sp.cos(O)) / (1 - e**2)

# Transverse kinematic projection (beta_T)
beta_T_shape = (1 + e * sp.cos(O)) / e_Y

# 3. Construct the 3rd-order transverse aberration mismatch (kappa^2 * beta_T^3)
integrand = kappa_sq_shape * (beta_T_shape)**3

print("--- 1. EXACT TRANSVERSE INTEGRAND ---")
print(sp.simplify(integrand))
print("\n")

# 4. Execute the analytic integration over the full 2*pi phase cycle
print("--- 2. NATIVE R.O.M. ECCENTRICITY SCALAR ---")
cumulative_shape = sp.integrate(integrand, (O, 0, 2*sp.pi))

# Isolate the polynomial structure by factoring out the constants
# The denominator is (1 - e^2)^(5/2) based on the geometric inputs
denominator = (1 - e**2)**(sp.Rational(5, 2))
numerator = sp.simplify(cumulative_shape * denominator / sp.pi)

print(f"Full integrated expression:\n{cumulative_shape}")
print(f"\nExtracted R.O.M. Shape Polynomial (normalized by pi):")
print(f"Numerator: {numerator}")
print(f"Denominator: (1 - e^2)^(5/2)")

--- 1. EXACT TRANSVERSE INTEGRAND ---
(e*cos(O) + 1)**4/(1 - e**2)**(5/2)


--- 2. NATIVE R.O.M. ECCENTRICITY SCALAR ---
Full integrated expression:
(3*pi*e**4/4 + 6*pi*e**2 + 2*pi)/(1 - e**2)**(5/2)

Extracted R.O.M. Shape Polynomial (normalized by pi):
Numerator: 3*e**4/4 + 6*e**2 + 2
Denominator: (1 - e^2)^(5/2)
